In [12]:
#imports
import os
import zipfile
from datetime import datetime
import requests
import pandas as pd
from google.transit import gtfs_realtime_pb2

In [13]:
#setup
BASE_DIR = "mta_project_data"
STATIC_DIR = f"{BASE_DIR}/gtfs_static"
REALTIME_DIR = f"{BASE_DIR}/gtfs_realtime_snapshots"

os.makedirs(STATIC_DIR, exist_ok=True)
os.makedirs(REALTIME_DIR, exist_ok=True)

# 1. Static GTFS Subway Dataset
STATIC_GTFS_URL = "https://rrgtfsfeeds.s3.amazonaws.com/gtfs_subway.zip"

# 2. Realtime GTFS feeds
REALTIME_FEEDS = {
    "1234567S": "https://api-endpoint.mta.info/Dataservice/mtagtfsfeeds/nyct%2Fgtfs",
    "ACE": "https://api-endpoint.mta.info/Dataservice/mtagtfsfeeds/nyct%2Fgtfs-ace",
    "BDFM": "https://api-endpoint.mta.info/Dataservice/mtagtfsfeeds/nyct%2Fgtfs-bdfm",
    "G": "https://api-endpoint.mta.info/Dataservice/mtagtfsfeeds/nyct%2Fgtfs-g",
    "JZ": "https://api-endpoint.mta.info/Dataservice/mtagtfsfeeds/nyct%2Fgtfs-jz",
    "NQRW": "https://api-endpoint.mta.info/Dataservice/mtagtfsfeeds/nyct%2Fgtfs-nqrw",
    "L": "https://api-endpoint.mta.info/Dataservice/mtagtfsfeeds/nyct%2Fgtfs-l"
}

In [39]:
def download_static_gtfs():
    zip_path = f"{BASE_DIR}/gtfs_subway.zip"

    #print("Downloading static GTFS...")
    response = requests.get(STATIC_GTFS_URL)
    response.raise_for_status()

    with open(zip_path, "wb") as f:
        f.write(response.content)

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(STATIC_DIR)

    #print("Static GTFS downloaded and extracted.")


def load_static_tables():
    print("\nStatic GTFS tables:")
    dflist = []
    dfnames = []
    for file in os.listdir(STATIC_DIR):
        if file.endswith(".txt"):
            path = f"{STATIC_DIR}/{file}"
            df = pd.read_csv(path)
            dflist.append(df)
            dfnames.append(file)
            print(f"\n{file}")
            print(df.head())
            print("Columns:", list(df.columns))
    return dflist, dfnames


def collect_realtime_snapshot():
    all_rows = []

    for feed_name, url in REALTIME_FEEDS.items():
        #print(f"Collecting realtime feed: {feed_name}")

        response = requests.get(url)
        response.raise_for_status()

        feed = gtfs_realtime_pb2.FeedMessage()
        feed.ParseFromString(response.content)

        for entity in feed.entity:
            if entity.HasField("vehicle"):
                vehicle = entity.vehicle

                all_rows.append({
                    "feed_name": feed_name,
                    "entity_id": entity.id,
                    "trip_id": vehicle.trip.trip_id,
                    "route_id": vehicle.trip.route_id,
                    "direction_id": vehicle.trip.direction_id,
                    "start_time": vehicle.trip.start_time,
                    "start_date": vehicle.trip.start_date,
                    "schedule_relationship": vehicle.trip.schedule_relationship,
                    "current_stop_sequence": vehicle.current_stop_sequence,
                    "current_status": vehicle.current_status,
                    "stop_id": vehicle.stop_id,
                    "timestamp_unix": vehicle.timestamp,
                    "timestamp_readable": datetime.fromtimestamp(vehicle.timestamp) if vehicle.timestamp else None,
                    "vehicle_id": vehicle.vehicle.id,
                    "vehicle_label": vehicle.vehicle.label,
                    "latitude": vehicle.position.latitude,
                    "longitude": vehicle.position.longitude,
                    "bearing": vehicle.position.bearing,
                    "odometer": vehicle.position.odometer,
                    "speed": vehicle.position.speed,
                    "congestion_level": vehicle.congestion_level,
                    "occupancy_status": vehicle.occupancy_status
                })

    df = pd.DataFrame(all_rows)

    now = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_path = f"{REALTIME_DIR}/mta_realtime_snapshot_{now}.csv"

    df.to_csv(output_path, index=False)

    #print("\nRealtime snapshot saved:")
    #print(output_path)

    #print("\nRealtime table preview:")
    #print(df.head())

    #print("\nColumns:")
    #print(list(df.columns))
    #print(df.isna().sum())

    return df


if __name__ == "__main__":
    download_static_gtfs()
    dflist, dfnames = load_static_tables()
    realtime_df = collect_realtime_snapshot()


Static GTFS tables:

agency.txt
  agency_id                agency_name           agency_url   agency_timezone  \
0  MTA NYCT  MTA New York City Transit  http://www.mta.info  America/New_York   

  agency_lang  agency_phone  
0          en  718-330-1234  
Columns: ['agency_id', 'agency_name', 'agency_url', 'agency_timezone', 'agency_lang', 'agency_phone']

calendar.txt
  service_id  monday  tuesday  wednesday  thursday  friday  saturday  sunday  \
0     Sunday       0        0          0         0       0         0       1   
1   Saturday       0        0          0         0       0         1       0   
2    Weekday       1        1          1         1       1         0       0   

   start_date  end_date  
0    20260301  20260516  
1    20260301  20260516  
2    20260301  20260516  
Columns: ['service_id', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday', 'start_date', 'end_date']

calendar_dates.txt
Empty DataFrame
Columns: [service_id, date, exception_t

In [53]:
# TODO: add code for other stuff
num = 0
'''for df in dflist:
    print(df.isna().sum())
    print(num)
    num += 1'''
print(dfnames)
print(dfnames[6])
print(dflist[6])


#drop dfs 0, 1, 2

#keep dfs 3, 4, 5, 6, 7, 8 

['agency.txt', 'calendar.txt', 'calendar_dates.txt', 'routes.txt', 'shapes.txt', 'stops.txt', 'stop_times.txt', 'transfers.txt', 'trips.txt']
stop_times.txt
                                            trip_id stop_id arrival_time  \
0            AFA25GEN-1038-Sunday-00_000600_1..S03R    101S     00:06:00   
1            AFA25GEN-1038-Sunday-00_000600_1..S03R    103S     00:07:30   
2            AFA25GEN-1038-Sunday-00_000600_1..S03R    104S     00:09:00   
3            AFA25GEN-1038-Sunday-00_000600_1..S03R    106S     00:10:30   
4            AFA25GEN-1038-Sunday-00_000600_1..S03R    107S     00:12:00   
...                                             ...     ...          ...   
562750  SIR-FA2017-SI017-Weekday-08_147100_SI..N03R    S27N     25:03:00   
562751  SIR-FA2017-SI017-Weekday-08_147100_SI..N03R    S28N     25:06:00   
562752  SIR-FA2017-SI017-Weekday-08_147100_SI..N03R    S29N     25:08:00   
562753  SIR-FA2017-SI017-Weekday-08_147100_SI..N03R    S30N     25:10:00   
562754 